# DANDI 000004: AMulti-Database Demo

## Overview: Neuronal Electrophysiology Analysis Using [DANDI 000004](https://dandi.github.io/example-notebooks/#dandiset-000004)

Neuroscience is a vast field, subject to domain experts. As such, there are terabytes of publicly available datasets on the topic, running from literature review to experimental design and analysis. One such database containing these datasets is DANDI: Distributed Archives for Neurophysiology Data Integration. Considered the gold standard for electrophysiology, DANDI uses the .nwb format (Neurodata Without Borders, a proprietary data format, organized into “Dandisets”) that plays nicely with vector databases, graph databases, and tabular databases. We explain our proposed approach below, using Dandiset 000004, “A NWB-based dataset and processing pipeline of human single-neuron activity during a declarative memory task”.

More on the DANDI API: https://dandi.readthedocs.io/en/latest/modref/dandiapi.html

This notebook demonstrates querying data that was ingested from a single streamed NWB session
(DANDI dandiset 000004) into the following databases:

| Database | Type | Purpose |
|---|---|---|
| **Neo4j** | Graph | Brain hierarchy (Subject -> Session -> Neuron -> Region) |
| **PostgreSQL** | Tabular | Subjects, sessions, neurons, trials |

In [1]:
import os
from dotenv import load_dotenv

# sanity checks
load_dotenv()
# print(os.environ.get("PG_DSN"))
# print(os.environ.get("NEO4J_URI"))

True

In [2]:
# create tables from schema.sql
import psycopg, os
from dotenv import load_dotenv
load_dotenv()

schema_path = os.path.join(os.path.dirname(os.path.abspath('.')), 'experiments', 'database', 'schema.sql')
schema_path = 'schema.sql'

with open(schema_path) as f:
    sql = f.read()

with psycopg.connect(os.environ['PG_DSN']) as conn, conn.cursor() as cur:
    cur.execute(sql)
    conn.commit()

In [3]:
# Stream and ingest every NWB session from DANDI 000004 into Postgres + Neo4j.
# No files are downloaded — data is fetched on-demand via HTTP range requests.
# Already-ingested sessions are detected from the DB and skipped automatically.
%run -i "data/ingest.py"

Dandiset 000004: 87 NWB assets found, 0 already ingested.


--- sub-P10HMH/sub-P10HMH_ses-20060901_ecephys+image.nwb ---
Session: H10_7
[Postgres] Ingested H10_7: 38 neurons, 200 trials.
[Neo4j]   Ingested H10_7: 38 neurons.

--- sub-P14HMH/sub-P14HMH_ses-20070601_obj-7studd_ecephys+image.nwb ---
Session: H14_18
[Postgres] Ingested H14_18: 33 neurons, 150 trials.
[Neo4j]   Ingested H14_18: 33 neurons.

--- sub-P15HMH/sub-P15HMH_ses-20070901_obj-a5xz9n_ecephys+image.nwb ---
Session: H15_21
[Postgres] Ingested H15_21: 14 neurons, 150 trials.
[Neo4j]   Ingested H15_21: 14 neurons.

--- sub-P14HMH/sub-P14HMH_ses-20070601_obj-1t8wrd5_ecephys+image.nwb ---
Session: H14_17
[Postgres] Ingested H14_17: 23 neurons, 200 trials.
[Neo4j]   Ingested H14_17: 23 neurons.

--- sub-P11HMH/sub-P11HMH_ses-20061101_ecephys+image.nwb ---
Session: H11_9
[Postgres] Ingested H11_9: 25 neurons, 200 trials.
[Neo4j]   Ingested H11_9: 25 neurons.

--- sub-P15HMH/sub-P15HMH_ses-20070901_obj-bdd49u_ecephys+image.nwb

## Neo4j: Graph Queries

In [ ]:
from importlib import reload
import utils.neo4j as n4j
reload(n4j)

Node breakdown:

```sh
MATCH (s:Subject)-[:HAS_SESSION]->(sess:Session)-[:HAS_NEURON]->(n:Neuron)-[:LOCATED_IN]->(r:BrainRegion)
RETURN s, sess, n, r
```

![graph_overview](img/graph_overview.jpg)

In [ ]:
# graph node counts
summary = n4j.get_graph_summary()
print('Graph node counts:')
display(summary)

Graph node counts:


,labels,count
0,[Neuron],1839
1,[Session],87
2,[Subject],59
3,[BrainRegion],4


![graph_summary](img/graph_summary.jpg)

In [ ]:
# list available brain regions
brain_regions = n4j.get_brain_regions()
print('Brain regions:')
display(brain_regions)

Brain regions:


,brain_region,n_neurons
0,Right Amygdala,577
1,Left Amygdala,494
2,Right Hippocampus,403
3,Left Hippocampus,365


![brain_regions](img/brain_regions.jpg)

In [ ]:
# experiment flow
# subject, session, neuron, region in that order
experiment_flow = n4j.get_experiment_flow()
print('Experiment flow:')
display(experiment_flow)

Experiment flow:


,s,sess,n,r
0,{'subject_id': 'P10HMH'},"{'date': '2006-09-01 00:00:00-07:00', 'session...","{'db_id': 38, 'unit_index': 37, 'brain_region'...",{'name': 'Left Amygdala'}
1,{'subject_id': 'P10HMH'},"{'date': '2006-09-01 00:00:00-07:00', 'session...","{'db_id': 37, 'unit_index': 36, 'brain_region'...",{'name': 'Left Amygdala'}
2,{'subject_id': 'P10HMH'},"{'date': '2006-09-01 00:00:00-07:00', 'session...","{'db_id': 36, 'unit_index': 35, 'brain_region'...",{'name': 'Left Amygdala'}
3,{'subject_id': 'P10HMH'},"{'date': '2006-09-01 00:00:00-07:00', 'session...","{'db_id': 35, 'unit_index': 34, 'brain_region'...",{'name': 'Left Amygdala'}
4,{'subject_id': 'P10HMH'},"{'date': '2006-09-01 00:00:00-07:00', 'session...","{'db_id': 34, 'unit_index': 33, 'brain_region'...",{'name': 'Left Amygdala'}
...,...,...,...,...
95,{'subject_id': 'P14HMH'},"{'date': '2007-06-01 00:00:00-07:00', 'session...","{'db_id': 90, 'unit_index': 4, 'brain_region':...",{'name': 'Right Amygdala'}
96,{'subject_id': 'P14HMH'},"{'date': '2007-06-01 00:00:00-07:00', 'session...","{'db_id': 89, 'unit_index': 3, 'brain_region':...",{'name': 'Right Amygdala'}
97,{'subject_id': 'P14HMH'},"{'date': '2007-06-01 00:00:00-07:00', 'session...","{'db_id': 88, 'unit_index': 2, 'brain_region':...",{'name': 'Right Amygdala'}
98,{'subject_id': 'P14HMH'},"{'date': '2007-06-01 00:00:00-07:00', 'session...","{'db_id': 87, 'unit_index': 1, 'brain_region':...",{'name': 'Right Amygdala'}


![experiment_flow](img/experiment_flow.jpg)

In [ ]:
neuron_clusters = n4j.get_neuron_clusters()
print('Neuron clusters:')
display(neuron_clusters)

Neuron clusters:


,sess,n,r
0,"{'date': '2008-06-01 00:00:00-07:00', 'session...","{'db_id': 266, 'unit_index': 30, 'brain_region...",{'name': 'Left Amygdala'}
1,"{'date': '2018-12-01 00:00:00-08:00', 'session...","{'db_id': 1864, 'unit_index': 2, 'brain_region...",{'name': 'Left Amygdala'}
2,"{'date': '2018-12-01 00:00:00-08:00', 'session...","{'db_id': 1863, 'unit_index': 1, 'brain_region...",{'name': 'Left Amygdala'}
3,"{'date': '2018-12-01 00:00:00-08:00', 'session...","{'db_id': 1862, 'unit_index': 0, 'brain_region...",{'name': 'Left Amygdala'}
4,"{'date': '2018-04-01 00:00:00-07:00', 'session...","{'db_id': 1836, 'unit_index': 0, 'brain_region...",{'name': 'Left Amygdala'}
...,...,...,...
95,"{'date': '2018-01-01 00:00:00-08:00', 'session...","{'db_id': 1537, 'unit_index': 7, 'brain_region...",{'name': 'Left Amygdala'}
96,"{'date': '2018-01-01 00:00:00-08:00', 'session...","{'db_id': 1536, 'unit_index': 6, 'brain_region...",{'name': 'Left Amygdala'}
97,"{'date': '2018-01-01 00:00:00-08:00', 'session...","{'db_id': 1535, 'unit_index': 5, 'brain_region...",{'name': 'Left Amygdala'}
98,"{'date': '2018-01-01 00:00:00-08:00', 'session...","{'db_id': 1534, 'unit_index': 4, 'brain_region...",{'name': 'Left Amygdala'}


![neuron_clusters](img/neuron_clusters.jpg)

In [ ]:
multi_region_sessions = n4j.get_multi_region_sessions()
print('Multi-region sessions:')
display(multi_region_sessions)

Multi-region sessions:


,sess.session_id,regions,unitCount
0,CS29_69,"[Left Amygdala, Left Hippocampus, Right Hippoc...",64
1,CS33_76,"[Left Amygdala, Right Amygdala, Right Hippocam...",53
2,H27_58,"[Left Hippocampus, Right Hippocampus]",49
3,CS54_125,"[Left Amygdala, Right Amygdala, Left Hippocamp...",46
4,H28_48,"[Left Amygdala, Right Amygdala, Right Hippocam...",45
...,...,...,...
70,H47_92,"[Right Amygdala, Right Hippocampus]",7
71,CS53_123,"[Left Amygdala, Right Hippocampus]",6
72,T90_2003,"[Left Amygdala, Right Amygdala]",5
73,T88_2002,"[Left Hippocampus, Right Hippocampus]",4


![multi_region_sessions](img/multi_region_sessions.jpg)

## PostgreSQL: Neuron Firing Statistics

After getting the data from Neo4j, we can refine our queries (and questions) even more with additional Postgres transactions.

In [ ]:
# if you update any function in postgres.py
# re-run this cell to get the lastest version
from importlib import reload
import utils.postgres as pg
reload(pg)

<module 'utils.postgres' from 'c:\\Users\\69458\\Desktop\\ucsd-dsc202\\utils\\postgres.py'>

In [ ]:
# session overview
sessions = pg.get_session_summary()
print(f'Ingested sessions: {len(sessions)}')
display(sessions)

Ingested sessions: 87


c:\Users\69458\Desktop\ucsd-dsc202\utils\postgres.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,session_id,subject_id,age,sex,institution,session_date,n_neurons,n_trials
0,H44_68,P44HMH,57,F,Hunington Memorial Hospital,2000-01-01 08:00:00+00:00,3800,3800
1,H09_6,P9HMH,55,M,Hunigton Memorial Hospital,2006-03-01 08:00:00+00:00,3800,3800
2,H09_5,P9HMH,55,M,Hunigton Memorial Hospital,2006-03-01 08:00:00+00:00,4800,4800
3,H10_7,P10HMH,37,M,Hunigton Memorial Hospital,2006-09-01 07:00:00+00:00,7600,7600
4,H11_9,P11HMH,16,M,Hunigton Memorial Hospital,2006-11-01 07:00:00+00:00,5000,5000
...,...,...,...,...,...,...,...,...
82,CS60_134,P60CS,67,M,Cedars-Sinai Medical Center,2018-10-01 07:00:00+00:00,2800,2800
83,T103_2009,TWH103,49,M,Toronto Western Hospital,2018-11-01 07:00:00+00:00,600,600
84,T107_2007,TWH107,64,F,Toronto Western Hospital,2018-12-01 08:00:00+00:00,600,600
85,CS61_135,P61CS,52,F,Cedars-Sinai Medical Center,2019-02-01 08:00:00+00:00,2600,2600


In [ ]:
#region summary
region_summary = pg.region_firing_summary()
print('Firing summary by brain region:')
print(region_summary)

Firing summary by brain region:
        brain_region  n_neurons  n_sessions  avg_firing_rate  min_firing_rate  \
0      Left Amygdala        498          54           0.9995           0.0227   
1   Left Hippocampus        365          50           1.2730           0.0177   
2     Right Amygdala        594          59           1.3319           0.0102   
3  Right Hippocampus        407          53           1.7955           0.0436   

   max_firing_rate  
0           9.2423  
1          13.2633  
2          11.4970  
3          16.9001  


c:\Users\69458\Desktop\ucsd-dsc202\utils\postgres.py:68: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  """


In [ ]:
#top neurons by region
top_region_neurons = pg.region_ranked_neurons(top_n=5)
print('Top 5 neurons by firing rate within each brain region:')
print(top_region_neurons)

Top 5 neurons by firing rate within each brain region:
         brain_region session_id  neuron_id  mean_firing_rate  region_rank
0       Left Amygdala    CS27_67        529            9.2423            1
1       Left Amygdala    CS27_67        528            7.6740            2
2       Left Amygdala    CS34_85        930            7.5233            3
3       Left Amygdala  T107_2007       1864            7.4208            4
4       Left Amygdala   CS40_101       1082            6.3050            5
5    Left Hippocampus     H48_99       1374           13.2633            1
6    Left Hippocampus    CS24_60        455           12.4406            2
7    Left Hippocampus     H33_52        912           12.1241            3
8    Left Hippocampus     H14_18         71           10.5737            4
9    Left Hippocampus   CS43_113       1174            8.5779            5
10     Right Amygdala     H14_17         86           11.4970            1
11     Right Amygdala   CS38_102        978  

c:\Users\69458\Desktop\ucsd-dsc202\utils\postgres.py:102: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.


In [ ]:
#UNNEST spike times
region_spikes = pg.region_spike_distribution()
print('Spike-time distribution by brain region (using UNNEST):')
print(region_spikes)

c:\Users\69458\Desktop\ucsd-dsc202\utils\postgres.py:133: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.


Spike-time distribution by brain region (using UNNEST):
        brain_region  total_spikes  n_neurons  avg_spike_time  \
0      Left Amygdala        950347        498    5.716904e+08   
1   Left Hippocampus        950524        365    4.582470e+08   
2     Right Amygdala       1488370        594    3.697457e+08   
3  Right Hippocampus       1048668        407    5.717559e+08   

   first_spike_time  last_spike_time  
0          846.5094     1.550664e+09  
1          763.9345     1.540811e+09  
2          914.7996     1.555517e+09  
3          895.8234     1.555517e+09  


In [ ]:
#session-vs-region comparison
region_session_scores = pg.region_session_zscores()
print('Session-level firing deviations within each brain region:')
print(region_session_scores)

Session-level firing deviations within each brain region:
          brain_region session_id  session_avg_rate  region_mean  \
0        Left Amygdala     H19_28            4.3070       1.2317   
1        Left Amygdala   CS57_130            4.0611       1.2317   
2        Left Amygdala  T107_2007            3.5269       1.2317   
3        Left Amygdala   CS38_102            3.0981       1.2317   
4        Left Amygdala     H48_99            2.5597       1.2317   
..                 ...        ...               ...          ...   
211  Right Hippocampus     H17_32            0.3574       1.6030   
212  Right Hippocampus     H23_47            0.2760       1.6030   
213  Right Hippocampus     H44_68            0.2669       1.6030   
214  Right Hippocampus     H21_38            0.2290       1.6030   
215  Right Hippocampus     H21_39            0.2290       1.6030   

     z_score_within_region  
0                   3.2438  
1                   2.9844  
2                   2.4209  
3        

c:\Users\69458\Desktop\ucsd-dsc202\utils\postgres.py:177: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
